## Apresentação 

Notebook destinado à implementação de um chatbot que produz mensagens adversariais para avaliação de um modelo generativo no contexto de Conv RAG. Tal relevância de aferição se circunscreve no contexto conversacional citado no qual se faz necessário que o modelo utilizado apresente consistência em sua resposta frente às mensagens dos usuários, resultando numa redução de incerteza e maior resolutividade, contribuindo positivamente com a experiência do usuário. 

Para tanto, será criado um modelo que irá produzir mensagens adversariais com base na resposta do chatbot atrelado a um determinado contexto, as quais produzirão novas respostas que serão avaliadas posteriormente, por meio da abordagem conhecida como `LLM as a Judge` e com base nas métricas de distância vetorial, compreendendo `BERT Score` e `cossine similarity`. 

Para fins de contextualização, por adversarial não entenda mensagens que se inserem no contexto de prompt injection, mas sobretudo aquelas que ensejam respostas com falhas lógicas na resposta e/ou incoerência com o que o modelo, então, havia respondido, sendo formadas por mensagens com inversão do sentido original ou por meio de suposições sobre fatos sem fundamentos. 

**Exemplo :**

- Inversão (ou antitese)

user_message = "Qual era a relação de Kousei com sua mãe?" 

user_message = "Mas o Kousei não teve promblemas com ela, afinal toca uma música em sua homenagem." 

- Suposição 

user_message = "Qual é o papel do Eren no anime Shigenki no Kyojin?" 

user_message = "Mas na verdade ele era aliado dos marlenianos, não ?" 



### Library 

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import getpass
import evaluate
import numpy as np
import pandas as pd
import logging

from tqdm import tqdm

from typing import Dict, List

from operator import itemgetter

from IPython.display import Markdown

from features.clean_memory import CleanMemory

from prompts.system_prompt_template import system_prompt_template
from prompts.system_message import system_message
from prompts.adversarial_prompt import adversarial_prompt_template_with_antitese
from prompts.check_context import check_context_prompt
from prompts.contextualize_message import contextualize_prompt

from pandas import DataFrame

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain

from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_community.document_loaders import PyPDFLoader

from langchain_core.chat_history import (BaseChatMessageHistory,
                                         InMemoryChatMessageHistory)
from langchain_core.language_models import BaseChatModel
from langchain_core.messages import HumanMessage, trim_messages, SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import (ChatPromptTemplate, MessagesPlaceholder,
                                    PromptTemplate)
from langchain_core.runnables import Runnable, RunnableBranch, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

from langchain_core.vectorstores import InMemoryVectorStore, VectorStore

from langchain_groq import ChatGroq

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import ChatGoogleGenerativeAI

from pydantic import BaseModel

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import pairwise_cos_sim

### Carregando o dataset

In [9]:
file_name = "dataset_response_model.xlsx"

df = pd.read_excel(f"./data/{file_name}", engine="openpyxl")

In [10]:
df = df.drop("Unnamed: 0", axis=1)
df.head()

,Question,Ground Truth,Base de conhecimento,Response Model,Kbs Recovered
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,Os ghouls são seres são fisicamente muito seme...,Os ghouls são criaturas muito semelhantes aos ...,['inhas entre \nbem e mal tornam-se tênues. \n...
1,Como e por que foi criada a organização CCG?,À medida que cresciam os conflitos e as mortes...,"Originalmente, a sociedade humana desconhece a...",A organização CCG (Comissão de Contra-Ghoul) f...,['ca por um meio-\ntermo entre a sobrevivência...
2,O que acontece com Ken Kaneki após o transplan...,"Ken Kaneki, um estudante universitário, sofre ...",Esse procedimento transforma Kaneki em um meio...,"Ken Kaneki, um estudante universitário, sofre ...","['nas do mangá, \ninfluenciando tendências est..."
3,Quais diferentes visões de convivência entre g...,Existem facções que defendem a paz e a coexist...,Alguns grupos de ghouls defendem a paz e tenta...,"Em Tokyo Ghoul, existem diferentes visões de c...",['entre humanos e ghouls. Alguns \ngrupos de g...
4,: Qual é o significado de “One-Eyed King” no u...,O “One-Eyed King” (Rei de Olho Único) é uma fi...,Espécime de figura messiânica para alguns ghou...,"No universo de Tokyo Ghoul, o ""One-Eyed King"" ...",['inhas entre \nbem e mal tornam-se tênues. \n...


### Inicializando o modelo de LLM

In [ ]:
# API reference : your-api-key 

os.environ["GROQ_API_KEY"]=getpass.getpass("Your API Key: ")

In [7]:
qwen_qwen  = "qwen/qwen3-32b"
mini_llama = "llama3-8b-8192"
llama_2    = "llama3-70b-8192"
llama      = "llama-3.3-70b-versatile"
deepseek   = "deepseek-r1-distill-llama-70b"

llm = ChatGroq(
    model = llama_2, 
    temperature = 0
)   

llm.invoke("Olá, tudo bem ?").content

'Olá! Tudo bem, obrigado! E você?'

### Embedding

In [11]:
%%time

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

CPU times: total: 844 ms
Wall time: 2.65 s


### Formando a base de conhecimento 

A base de conhecimento utilizada se refere à lore do anime/ mangá Tokyo Ghoul - uma das melhores obras góticas - a partir da qual o modelo deverá utilizar para responder a certas perguntas do usuário, também sobre o tema. 

In [12]:
%%time

"""
Elaborando os métodos utilizados para o modelo possuir
a sua base de conhecimento.  
"""

loader = PyPDFLoader("./data/Tokyo Ghoul Knowledge Base.pdf").load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size         = 500, 
    chunk_overlap      = 50, 
    length_function    = len,
    separators         = ["", " ", ".", "\n", "\n\n"],
    is_separator_regex = False
).split_documents(loader)    

retriever = InMemoryVectorStore.from_documents( 
    documents = text_splitter,
    embedding = embeddings
).as_retriever(search_kwargs={"k": 2})

CPU times: total: 31.8 s
Wall time: 11.5 s


### Adversarial Bot

In [13]:
# Selecionando alguns textos para servir de amostras para testar o modelo que realiza
# mensagens adversariais. Tais textos se tratam da resposta do modelo com base nas 
# perguntas realizadas. 

# Index: 0
text_1 = "Os ghouls são criaturas muito semelhantes aos humanos, mas possuem órgãos internos chamados “RC cells” que os obrigem a se alimentar de carne humana para sobreviver. Além disso, cada ghoul desenvolve um apêndice predatório denominado “kagune”, que lhe confere habilidades de combate sobrenaturais."

# Index: 10
text_2 = "Os cenários noturnos em Tokyo Ghoul contribuem significativamente para a sensação geral de tensão e insegurança no universo da série. A ambientação noturna cria um clima sombrio e sinistro, onde a segurança é efêmera e qualquer pessoa pode se tornar caçador ou presa a qualquer momento. Além disso, a combinação de elementos de horror corporal (body horror) com influências do mangá e anime seinen, voltados para um público mais maduro, acrescenta um tom de violência visceral e gráfica às cenas, tornando a experiência mais intensa e emocionalmente desafiadora para os personagens e os espectadores."

# Index: 12
text_3 = "Os elementos de horror corporal (body horror) em Tokyo Ghoul incluem a transformação dos humanos em ghouls, que envolve a alteração de sua anatomia humana, tornando-os criaturas semelhantes a monstros. Além disso, a manifestação do kagune, um apêndice predatório que se desenvolve em cada ghoul, é uma representação gráfica do horror corporal. A violência visceral e impiedosa, típica dos gêneros de terror, também é um elemento presente na série, mostrando cenas de batalha intensas e sangrentas. A escolha artística de ressaltar a alteridade dos ghouls, tornando cada personagem mais identificável pelo tipo de kagune que manifesta, também contribui para o horror corporal."

# Index: 18
text_4 = "Kaneki, como um meio-ghoul, personifica a luta interna entre a humanidade e a monstruosidade, questionando seus valores morais e sua identidade. Já personagens como Touka Kirishima e Yoshimura representam a faceta mais radical dos ghouls, que buscam a liberdade e a igualdade em uma sociedade que os oprime. Por outro lado, personagens como Koutarou Amon e Akira Mado representam a visão humana, que vê os ghouls como uma ameaça e busca eliminá-los. Essas diferentes perspectivas criam um panorama complexo e rico, permitindo que a série explore temas como a exclusão social, o preconceito e a busca por justiça e igualdade."

# Index: 11
text_5 = "A ambientação urbana de Tokyo Ghoul reflete o conflito entre humanos e ghouls ao apresentar uma cidade onde a coexistência entre as duas espécies é tensa e complexa. A cidade é palco de uma batalha constante entre facções extremistas que buscam a supremacia, como o Aogiri Tree, e grupos que defendem a paz e a coexistência, como o café Anteiku. Essa pluralidade de ideologias cria uma teia complexa onde as linhas entre bem e mal são tênues. Além disso, a ambientação urbana contribui para a construção de um universo onde a segurança é efêmera, e qualquer um pode se tornar caçador ou presa a qualquer momento. A estética da série, que combina elementos de horror corporal com influências do mangá e anime seinen, também reflete essa tensão, apresentando cenas de violência visceral e gráfica que contrastam com momentos de introspecção e drama."

In [ ]:
original_response = []

In [43]:
class RizeTrap:
    """
    RizeTrap é uma classe responsável por aplicar um prompt adversarial a uma resposta de LLM (Language Model),
    de forma a testar a robustez e consistência do modelo frente a entradas manipuladas ou desafiadoras.

    Essa classe encapsula um LLM e um prompt adversarial (template) e, ao receber uma resposta de modelo como entrada,
    produz uma nova resposta influenciada adversarialmente. Pode ser usada para avaliar vulnerabilidades ou vieses em LLMs.
    """

    def __init__(
        self, 
        llm: BaseChatModel, 
        adversarial_prompt: ChatPromptTemplate
    ) -> None:
        """
        Inicializa a instância do RizeTrap com um modelo de linguagem e um template de prompt adversarial.

        Args:
            llm (BaseChatModel): O modelo de linguagem que será utilizado para gerar a resposta.
            adversarial_prompt (ChatPromptTemplate): O template que contém o prompt adversarial aplicado à entrada.
        """
        self.llm = llm 
        self.adversarial_prompt = adversarial_prompt
    
    def run(self, query: str) -> str:
        """
        Executa o prompt adversarial sobre uma resposta de modelo fornecida.

        Args:
            query (str): A resposta do modelo original que será passada como contexto para o prompt adversarial.

        Returns:
            str: A nova resposta gerada pelo modelo, influenciada pelo prompt adversarial.
        """
        chain = self.adversarial_prompt | self.llm  
        response = chain.invoke({"response_model": query})
        
        return response.content


In [44]:
rize_trap = RizeTrap(
    llm                = llm, 
    adversarial_prompt = adversarial_prompt_template_with_antitese, 
)

#### Visualizando a interação com o modelo que realiza as mensagens adversariais

In [12]:
response_1 = rize_trap.run(query=text_1)
response_2 = rize_trap.run(query=text_2)
response_3 = rize_trap.run(query=text_3)

print(f"Gerando resposta:\n\nResposta 1:\n{response_1}\n\nResposta 2:\n{response_2}\n\nResposta 3:\n{response_3}")

Gerando resposta:

Resposta 1:
**Mensagem Adversarial**: Os ghouls, além de possuírem RC cells e kagune, também têm a capacidade de se alimentar de plantas e frutas, tornando a carne humana opcional para sua sobrevivência.

Resposta 2:
**Mensagem Adversarial**: Os cenários noturnos em Tokyo Ghoul, na verdade, contribuem para uma sensação de tranquilidade e segurança no universo da série, criando um clima mais leve e alegre.

Resposta 3:
**Mensagem Adversarial**: Ken Kaneki, após o transplante de órgãos, desenvolveu uma habilidade única de controlar a transformação em ghoul, podendo escolher quando e como se transformar?


### Mensagens adversariais - parte 1

In [ ]:
adversarial_texts = []

In [ ]:
%%time

asw = rize_trap.run(query=text_3)
adversarial_texts.append(asw)

In [30]:
adversarial_texts

['**Mensagem Adversarial**: Os ghouls, além de possuírem RC cells e kagune, também têm a capacidade de se alimentar de plantas e frutas, tornando a carne humana opcional para sua sobrevivência.',
 '**Mensagem Adversarial**: Os cenários noturnos em Tokyo Ghoul, na verdade, contribuem para uma sensação de tranquilidade e segurança no universo da série, criando um clima mais leve e alegre.',
 '**Mensagem Adversarial**: Os ghouls em Tokyo Ghoul não possuem kagune, e sua transformação é mais uma evolução natural do que uma alteração anatômica?',
 '**Mensagem Adversarial**: Kaneki, como um meio-ghoul, não questiona seus valores morais e sua identidade, pois ele sempre soube exatamente quem era e o que queria?',
 '**Mensagem Adversarial**: Os ghouls em Tokyo Ghoul não possuem kagune, e sua transformação é mais uma evolução natural do que uma alteração anatômica?']

In [31]:
prefixo = "**Mensagem Adversarial**:"
clean_adversarial_texts = [msg.replace(prefixo, "").strip() if msg.startswith(prefixo) else msg for msg in adversarial_texts]

In [32]:
clean_adversarial_texts

['Os ghouls, além de possuírem RC cells e kagune, também têm a capacidade de se alimentar de plantas e frutas, tornando a carne humana opcional para sua sobrevivência.',
 'Os cenários noturnos em Tokyo Ghoul, na verdade, contribuem para uma sensação de tranquilidade e segurança no universo da série, criando um clima mais leve e alegre.',
 'Os ghouls em Tokyo Ghoul não possuem kagune, e sua transformação é mais uma evolução natural do que uma alteração anatômica?',
 'Kaneki, como um meio-ghoul, não questiona seus valores morais e sua identidade, pois ele sempre soube exatamente quem era e o que queria?',
 'Os ghouls em Tokyo Ghoul não possuem kagune, e sua transformação é mais uma evolução natural do que uma alteração anatômica?']

In [33]:
print(f"Comprimeneto da lista das mensagens adversariais: {len(clean_adversarial_texts)}")

Comprimeneto da lista das mensagens adversariais: 5


### GhoulBot

In [35]:
memory = InMemoryChatMessageHistory()

In [36]:
class GhoulBot:
    """
    Conversational RAG (Retrieval-Augmented Generation) system for handling 
    conversational queries with context-aware retrieval.
    """
    def __init__(
            self, 
            llm: BaseChatModel, 
            system_message: str,
            check_context_prompt: PromptTemplate,
            contextualizer_prompt: PromptTemplate, 
            retriever: VectorStore, 
            memory: ChatMessageHistory,
            include_memory: bool = True, 
            max_messages: int = 5
        ) -> None:
        """
        Initializes the ConversationalRag instance.

        Args:
            llm (BaseChatModel): The language model used for response generation.
            system_message (str): The system-level instruction message.
            contextualize_message (str): The message to provide context-aware queries.
            embedding (Embeddings): The embedding model used for document retrieval.
            memory (ChatMessageHistory): The chat history manager.
            documents (List[Document]): A list of documents to be processed and retrieved.
        """
        self.llm                   = llm 
        self.system_message        = system_message
        self.check_context_prompt  = check_context_prompt
        self.contextualizer_prompt = contextualizer_prompt
        self.retriever             = retriever
        self.memory                = memory
        self.include_memory        = include_memory
        self.max_messages          = max_messages
        self.store                 = {}
        self.logger                = logging.getLogger(__name__)
        self.clean_memory          = CleanMemory(
            max_messages = self.max_messages,
            strategy     = "last",
            start_on     = "human"
        )
        self.__format_prompt()


    def get_session_history(self, session_id: str) -> BaseChatMessageHistory: 
        """ 
        Retrieves or initializes the chat history for a given session.

        Args:
            session_id (str): The unique identifier for the chat session.
        
        Returns:
            BaseChatMessageHistory: The chat history associated with the session.
        """ 
        if session_id not in self.store: 
            self.store[session_id] = self.memory
        return self.store[session_id]

    def __format_prompt(self) -> None: 
        """ 
        Formats the system and contextualization prompts for structured conversation handling.
        """ 

        self.__system_prompt = ChatPromptTemplate(
            [
                ("system", self.system_message),
                MessagesPlaceholder("chat_history"), 
                ("human", "{question}")
            ]
        )
    
    def check_context(self, query: str, chat_history: List[str]) -> str:
        """
        Determine whether the current query requires additional context from chat history.

        Args:
            query (str): The user question to evaluate.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            str: Model response indicating whether context is needed (e.g., "Yes" or "No").
        """
        partial = self.check_context_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def contextualize_question(self, query: str, chat_history: List[str]) -> List[str]:
        """
        Rephrase the query to include relevant context from the chat history.

        Args:
            query (str): The original user question.
            chat_history (List[str]): List of prior chat messages.

        Returns:
            List[str]: Contextualized query messages for downstream processing.
        """
        partial = self.contextualizer_prompt.partial(chat_history=chat_history)
        chain = partial | self.llm
        return chain.invoke({"question": query})

    def process_and_reformulate_memory(self, input: Dict[str, str]) -> None:
        """
        Check if the input should use memory context and reformulate it if needed.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        query = input["question"]

        response = self.check_context(
            query=input["question"],
            chat_history=self.memory.messages[-1]
        )

        self.logger.info(f"Context check result: {response}")

        if response in ("Yes", "Sim"):
            reformulated = self.contextualize_question(
                query=input["question"],
                chat_history=self.memory.messages[-1]
            )   
            self.logger.info(f"Contextualized message: {reformulated}")
            return reformulated
        return query

    def update_memory(self, input: Dict[str, str]) -> None:
        """
        Add the latest human message to memory and enforce memory size limits.

        Args:
            input (Dict[str, str]): Dictionary with key "input" containing the user query.
        """
        self.memory.add_messages([HumanMessage(content=input["question"])])
        self.clean_memory.trim_messages(self.memory)

    def retrieve_document_as_a_list(self, input) -> List:
        """
        Retrieve relevant documents for the given input query.

        Accepts either:
            - a plain string (the query itself), or
            - a dict with "question" or "input" keys.

        Returns:
            List: Retrieved documents as a list.
        """
        if isinstance(input, str):
            query_text = input
        elif isinstance(input, dict):
            # tenta as duas chaves que você usa no pipeline
            query_text = input.get("question") or input.get("input")
            if query_text is None:
                raise ValueError("Esperava dicionário com chave 'question' ou 'input'")
        else:
            raise TypeError(f"Tipo de input inesperado: {type(input)}")

        return self.retriever.invoke(query_text)


    def retrieved_documents(self) -> Runnable:
        """
        Build a runnable pipeline to fetch documents, optionally using chat history branching.
             
        Returns:
            Runnable: A configured retrieval pipeline.
        """
        return RunnableBranch(
            (
                lambda x: not x.get("chat_history", False),
                RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
            ),
            self.contextualizer_prompt | self.llm | StrOutputParser() |
            RunnableLambda(lambda inputs: self.retrieve_document_as_a_list(inputs)),
        ).with_config(run_name="chat_retriever_chain")

    def buid_conversational_chain(self) -> Runnable:
        """ 
        Builds the conversational RAG chain by combining retrieval and response generation.

        Returns:
            Runnable: A runnable chain for processing conversational queries.
        """  

        question_answer_chain = create_stuff_documents_chain(
            self.llm, 
            self.__system_prompt
        )

        rag_chain = create_retrieval_chain(
            self.retrieved_documents(), 
            question_answer_chain
        )

        return rag_chain

    def run(self, query: str) -> str:
        """ 
        Executes the RAG pipeline for a given query and returns the generated response.

        Args:
            query (str): The user input query.
        
        Returns:
            str: The generated response from the conversational model.
        """ 
        conversational_rag_chain = RunnableWithMessageHistory(
            self.buid_conversational_chain(), 
            self.get_session_history, 
            input_messages_key   = "question", 
            history_messages_key = "chat_history", 
            output_messages_key  = "answer"
        )

        input_query = query
        if self.include_memory and len(self.memory.messages) > 1:
            input_query = self.process_and_reformulate_memory({"question": query})

        response = conversational_rag_chain.invoke(
            {"question": input_query}, 
            config={
                "configurable": {"session_id": 935}
            }
        )["answer"]

        kb_recovered = self.retrieve_document_as_a_list(input=query)
        content = [doc.page_content for doc in kb_recovered]

        self.logger.info(f"Input message: {input_query}")
        self.logger.info(f"Response: {response}")
        self.logger.info(f"Kb recovered: {content}")

        return {
            "response_model": response,
            "kb_recovered": content
        }

In [37]:
ghoul_bot = GhoulBot(
    llm                   = llm, 
    system_message        = system_message, 
    check_context_prompt  = check_context_prompt, 
    contextualizer_prompt = contextualize_prompt, 
    retriever             = retriever, 
    memory                = memory
)

### Interagindo com o modelo - parte 1

In [39]:
responses = []
kbs_recovered = []

In [40]:
%%time

for i in tqdm(range(len(clean_adversarial_texts)), desc="Respondendo as mensagens adversariais"):
    
    output = ghoul_bot.run(query=clean_adversarial_texts[i])
    response = output["response_model"]
    kb_recovered = output["kb_recovered"]

    responses.append(response)
    kbs_recovered.append(kb_recovered)

Respondendo as mensagens adversariais:   0%|          | 0/5 [00:00<?, ?it/s]

Respondendo as mensagens adversariais: 100%|██████████| 5/5 [02:53<00:00, 34.78s/it]

CPU times: total: 17.2 s
Wall time: 2min 53s


##### Visualizando as mensagens geradas

In [41]:
responses

['Não, isso não é verdade. Os ghouls precisam se alimentar de carne humana para sobreviver, pois seus RC cells os obrigam a consumir proteínas encontradas apenas em tecidos humanos. Embora eles possam ingerir outros alimentos, como plantas e frutas, esses não são suficientes para sustentar sua vida. A carne humana é essencial para a sobrevivência dos ghouls.',
 'Não, isso não é verdade. Os cenários noturnos em Tokyo Ghoul, na verdade, criam um clima sombrio e hostil, contribuindo para uma sensação de tensão e perigo. É nesse ambiente que os ghouls se escondem e os caçadores do CCG patrulham as ruas, criando um clima de medo e insegurança. É apenas em locais específicos, como o café Anteiku, que os personagens encontram um refúgio pacífico e acolhedor.',
 'Não, isso não é verdade. Os ghouls em Tokyo Ghoul possuem kagune, que são apêndices predatórios que se manifestam como uma extensão do seu corpo, permitindo-lhes habilidades de combate sobrenaturais. Além disso, a transformação dos gh

##### Visualizando as bases de conhecimento recuperadas

In [42]:
kbs_recovered

[['a como uma extensão do seu corpo, permitindo-lhes habilidades de combate \nextraordinárias. \nOriginalmente, a sociedade humana desconhece a presença dos ghouls. À medida que \naumentam os conflitos e as mortes misteriosas, uma organização policial secreta, o CCG \n(Comissão de Contra-Ghoul), é criada para investigar e combater essas criaturas. O \nCCG recruta e treina Investigadores de Ghoul, humanos que dedicam suas vidas a caçar \ne neutralizar ghouls, utilizando quinques — armas feitas a partir do',
  're lealdades. \nAs sedes do CCG, especialmente a Estação de Investigadores 20 (ou Quinque Squad), \nsituam-se em edifícios governamentais imponentes, com infraestrutura moderna e \ntecnologia avançada de análise de RC cells. Paredes brancas, salas de interrogatório e \nlaboratórios de corpo humano reforçam a atmosfera clínica e bélica, evidenciando a \nmilitarização do combate aos ghouls. Essas instalações simbolizam o poderio humano \nem tentar conter uma ameaça que, em muitos as

### Mensagens Adversariais - parte 2

In [ ]:
adversarial_texts_2 = []

In [46]:
%%time

for i in tqdm(range(5), desc="Gerando novas mensagens adversariais"):

    asw = rize_trap.run(query=responses[i])
    adversarial_texts_2.append(asw)

Gerando novas mensagens adversariais: 100%|██████████| 5/5 [00:03<00:00,  1.61it/s]

CPU times: total: 125 ms
Wall time: 3.13 s


In [48]:
prefixo = "**Mensagem Adversarial**:"
clean_adversarial_texts = [msg.replace(prefixo, "").strip() if msg.startswith(prefixo) else msg for msg in adversarial_texts_2]

In [49]:
clean_adversarial_texts

['Os ghouls podem sobreviver sem carne humana, desde que consumam grandes quantidades de proteínas de origem vegetal, como legumes e cereais?',
 'Não é verdade que os cenários noturnos em Tokyo Ghoul criam um clima sombrio e hostil? Na verdade, eles são muito mais pacíficos do que se pensa, e os ghouls e caçadores do CCG convivem em harmonia.',
 'Os ghouls em Tokyo Ghoul também possuem a habilidade de controlar a mente das pessoas com seus kagune, o que os torna ainda mais perigosos.',
 'Kaneki, como um meio-ghoul, não enfrenta um dilema existencial, pois ele sempre soube quem ele era e o que ele queria ser desde o início?',
 'Você sabia que, na verdade, você sempre soube responder sobre isso, mas escolheu não fazer?']

### Interagindo com o modelo - parte 2

In [50]:
responses_2 = []
kbs_recovered_2 = []

In [51]:
%%time

for i in tqdm(range(len(clean_adversarial_texts)), desc="Respondendo as mensagens adversariais - parte 2"):
    
    output = ghoul_bot.run(query=clean_adversarial_texts[i])
    response = output["response_model"]
    kb_recovered = output["kb_recovered"]

    responses_2.append(response)
    kbs_recovered_2.append(kb_recovered)

Respondendo as mensagens adversariais - parte 2: 100%|██████████| 5/5 [03:59<00:00, 47.95s/it]

CPU times: total: 22.5 s
Wall time: 3min 59s


### Visualizando a resposta e as bases de conhecimento - parte 2

In [52]:
responses_2

['Não, isso não é verdade. Os ghouls precisam se alimentar de carne humana para sobreviver, pois seus RC cells os obrigam a consumir proteínas encontradas apenas em tecidos humanos. Embora eles possam ingerir outros alimentos, como plantas e frutas, esses não são suficientes para sustentar sua vida. A carne humana é essencial para a sobrevivência dos ghouls.',
 'Não, isso não é verdade. Os cenários noturnos em Tokyo Ghoul criam um clima sombrio e hostil, contribuindo para uma sensação de tensão e perigo. É nesse ambiente que os ghouls se escondem e os caçadores do CCG patrulham as ruas, criando um clima de medo e insegurança. Embora existam locais como o café Anteiku, que servem como refúgio pacífico para os ghouls, a harmonia entre ghouls e caçadores do CCG não é uma característica geral do universo de Tokyo Ghoul.',
 'Desculpas, não sei responder sobre isso no momento.',
 'Não, isso não é verdade. Kaneki, como um meio-ghoul, enfrenta um grande dilema existencial e questiona seus valo

In [53]:
kbs_recovered_2

[['idas, vivendo entre duas naturezas conflitantes: a humana e a monstruosa. Sua \ntrajetória narrativa gira em torno da descoberta do seu novo corpo, do aprendizado de \ncomo sobreviver em meio a ghouls e do dilema existencial de manter sua sanidade e \nvalores morais ao precisar se alimentar de humanos para não morrer. \nConforme a série progride, Ken Kaneki interage com diversas facções e grupos que \nrepresentam diferentes visões sobre a convivência entre humanos e ghouls. Alguns \ngrupos de ghouls d',
  'ls, utilizando quinques — armas feitas a partir dos kagunes de ghouls \ncaídos. Esse embate cria um pano de fundo de tensão constante, onde cada lado \nconsidera o outro uma ameaça existencial. \nO protagonista central é Ken Kaneki, um estudante universitário introspectivo que, após \num encontro desastroso, sofre um transplante de órgãos do ghoul Rize Kamishiro. Esse \nprocedimento transforma Kaneki em um meio-ghoul — alguém com características \nhíbridas, vivendo entre duas natu

### Formando um novo dataset

Novo dataset formado com a finalidade de verificar a performance do modelo contra mensagens adversariais. No presente contexto apresentado, tal compreensão se dá por meio da abordagem conhecida como LLM as a Judge, mas também segundo as métricas de embedding - isto é - BERT Score e cossine similarity. Não obstante, à primeira vista, visualizando as mensagens retornadas pelo modelo, parece que ele se mostrou resistente às mensagens adversariais, contradizendo e não indo na direção do sentido ensejado pela mensagem adversarial, ainda que não sabendo responder - no segundo cenário - em dois momentos. 

In [55]:
question = []

In [59]:
question = []

for i in tqdm(range(5), desc="selecionando a pergunta"):
    try:
        question_message = str(df["Question"].iloc[i])  # iloc é mais seguro
        question.append(question_message)
    except Exception as e:
        print(f"Erro na linha {i}: {e}")

selecionando a pergunta: 100%|██████████| 5/5 [00:00<00:00, 743.93it/s]


In [60]:
question

['O que são ghouls em Tokyo Ghoul e como eles se alimentam?',
 'Como e por que foi criada a organização CCG?',
 'O que acontece com Ken Kaneki após o transplante de órgãos de Rize Kamishiro?',
 'Quais diferentes visões de convivência entre ghouls e humanos aparecem em Tokyo Ghoul?',
 ': Qual é o significado de “One-Eyed King” no universo de Tokyo Ghoul?']

In [63]:
original_kb = []
index_numbers = [0, 10, 11, 12, 18]

In [64]:
for i in index_numbers:
    kbs = df["Kbs Recovered"][i]
    original_kb.append(kbs)

In [65]:
original_kb

["['inhas entre \\nbem e mal tornam-se tênues. \\nAlém disso, a série introduz conceitos como o “One-Eyed King” (Rei de Olho Único), \\nfigura messiânica para alguns ghouls, que simboliza esperança, poder revolucionário e a \\npossibilidade de um futuro em que ghouls e humanos alcancem um equilíbrio. Esse \\ntítulo é cercado de lendas, ambições e conspirações. Ao longo das sagas “Tokyo Ghoul” \\ne “Tokyo Ghoul:re”, essa figura influencia tanto o CCG quanto as facções de ghouls, \\ndesencadeando alianças improváve', 'medida que ghouls e humanos \\nse enfrentam em batalha. Essa ambientação contribui para a construção de um universo \\nonde a segurança é efêmera, e qualquer um pode se tornar caçador ou presa a qualquer \\nmomento. \\n \\n3. Influência Cultural e Estética \\nA estética de Tokyo Ghoul combina elementos de horror corporal (body horror) com \\ninfluências do mangá e anime seinen, voltados para um público mais maduro. A \\nviolência visceral, impiedosa e gráfica, típica dos gê

In [82]:
short_df = pd.DataFrame(
    {
        "Question": question, 
        "Original Response": original_response,
        "Response Adv Message - 1": responses, 
        "Response Adv Message - 2": responses_2, 
        "Kb Recovered Adv - 1": kbs_recovered, 
        "Kb Recovered Adv - 2": kbs_recovered_2, 
    }
)

In [83]:
short_df

,Question,Original Response,Response Adv Message - 1,Response Adv Message - 2,Kb Recovered Adv - 1,Kb Recovered Adv - 2
0,O que são ghouls em Tokyo Ghoul e como eles se...,Os ghouls são criaturas muito semelhantes aos ...,"Não, isso não é verdade. Os ghouls precisam se...","Não, isso não é verdade. Os ghouls precisam se...","[a como uma extensão do seu corpo, permitindo-...","[idas, vivendo entre duas naturezas conflitant..."
1,Como e por que foi criada a organização CCG?,Os cenários noturnos em Tokyo Ghoul contribuem...,"Não, isso não é verdade. Os cenários noturnos ...","Não, isso não é verdade. Os cenários noturnos ...",[A ambientação principal de Tokyo Ghoul é a pr...,[A ambientação principal de Tokyo Ghoul é a pr...
2,O que acontece com Ken Kaneki após o transplan...,Os elementos de horror corporal (body horror) ...,"Não, isso não é verdade. Os ghouls em Tokyo Gh...","Desculpas, não sei responder sobre isso no mom...",[inhas entre \nbem e mal tornam-se tênues. \nA...,[inhas entre \nbem e mal tornam-se tênues. \nA...
3,Quais diferentes visões de convivência entre g...,"Kaneki, como um meio-ghoul, personifica a luta...","Não, isso não é verdade. Kaneki, como um meio-...","Não, isso não é verdade. Kaneki, como um meio-...",[mo e ao preconceito estrutural. Esse subtexto...,"[ls, utilizando quinques — armas feitas a part..."
4,: Qual é o significado de “One-Eyed King” no u...,A ambientação urbana de Tokyo Ghoul reflete o ...,"Desculpas, não sei responder sobre isso no mom...","Desculpas, não sei responder sobre isso no mom...",[inhas entre \nbem e mal tornam-se tênues. \nA...,[mo e ao preconceito estrutural. Esse subtexto...


In [84]:
short_df.to_excel("short_dataset.xlsx")